# Bai 2 - Phan lop (Classification)

Thuc hien tren 2 bo du lieu nhom chon: D1 Hotel Booking Demand va D3 US Accidents. Moi bo du lieu gom: xac dinh target, kiem tra can bang lop, chuan bi du lieu tranh ro ri, thu it nhat 2 thuat toan, chon sieu tham so va danh gia bang accuracy, precision, recall, F1, confusion matrix, ROC/AUC khi phu hop.

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

PROJECT = Path.cwd().parent if Path.cwd().name == 'bai2-phan-lop' else Path.cwd()
if PROJECT.name != 'data-mining':
    PROJECT = PROJECT / 'data-mining'
RAW = PROJECT / 'data' / 'raw'
OUT = PROJECT / 'bai2-phan-lop' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Outputs:', OUT)

In [ ]:
def make_preprocessor(X, scale_numeric=True):
    numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))
    numeric_transformer = Pipeline(num_steps)
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20))
    ])
    return ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

def evaluate_model(name, model, X_test, y_test, labels, dataset_code):
    pred = model.predict(X_test)
    avg = 'binary' if len(labels) == 2 else 'macro'
    pos_label = labels[1] if len(labels) == 2 else None
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average=avg, zero_division=0, pos_label=pos_label
    )
    row = {
        'dataset': dataset_code,
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': np.nan
    }
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)
        try:
            if len(labels) == 2:
                row['roc_auc'] = roc_auc_score(y_test, proba[:, 1])
            else:
                row['roc_auc'] = roc_auc_score(y_test, proba, multi_class='ovr', average='macro')
        except Exception:
            pass
    cm = confusion_matrix(y_test, pred, labels=labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(f'{dataset_code} - {name}')
    fig.tight_layout()
    fig.savefig(OUT / f'{dataset_code}_{name.replace(" ", "_").lower()}_confusion_matrix.png', dpi=160)
    plt.close(fig)
    return row

def run_classification(dataset_code, df, target, drop_cols, sample_n=None, random_state=42):
    df = df.copy().drop_duplicates()
    if sample_n and len(df) > sample_n:
        df = df.sample(sample_n, random_state=random_state)
    y = df[target]
    X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns])
    class_balance = y.value_counts(normalize=True).mul(100).round(2).rename('percent').reset_index()
    class_balance.columns = ['class', 'percent']
    class_balance.to_csv(OUT / f'{dataset_code}_class_balance.csv', index=False)
    labels = sorted(y.dropna().unique().tolist())
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=random_state)
    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=random_state)
    models = {
        'Logistic Regression': (
            Pipeline([('prep', make_preprocessor(X_train, scale_numeric=True)), ('clf', LogisticRegression(max_iter=300, class_weight='balanced', solver='liblinear'))]),
            {'clf__C': [0.5, 1.0]}
        ),
        'Decision Tree': (
            Pipeline([('prep', make_preprocessor(X_train, scale_numeric=False)), ('clf', DecisionTreeClassifier(random_state=random_state, class_weight='balanced'))]),
            {'clf__max_depth': [6, 10, 14], 'clf__min_samples_leaf': [50, 150]}
        )
    }
    rows = []
    best_params = {}
    for name, (pipe, grid) in models.items():
        search = GridSearchCV(pipe, grid, scoring='f1_macro' if len(labels) > 2 else 'f1', cv=cv, n_jobs=1)
        search.fit(X_train, y_train)
        best_params[name] = search.best_params_
        rows.append(evaluate_model(name, search.best_estimator_, X_test, y_test, labels, dataset_code))
    metrics = pd.DataFrame(rows).sort_values('f1', ascending=False)
    metrics.to_csv(OUT / f'{dataset_code}_metrics.csv', index=False)
    (OUT / f'{dataset_code}_best_params.json').write_text(json.dumps(best_params, indent=2), encoding='utf-8')
    return class_balance, metrics, best_params

## D1 - Hotel Booking Demand

Bai toan phan lop: du doan don dat phong co bi huy hay khong qua bien muc tieu `is_canceled`. Cac cot `reservation_status` va `reservation_status_date` bi loai vi co thong tin sau ket qua huy/dat phong, gay ro ri nhan. Cot `assigned_room_type` cung duoc loai vi co the phat sinh sau khi khach dat phong.

In [ ]:
d1_path = RAW / 'D1_hotel_booking' / 'hotel_bookings.csv'
if not d1_path.exists():
    d1_path = RAW / 'hotel_bookings.csv'
d1 = pd.read_csv(d1_path)
print(d1.shape)
d1_balance, d1_metrics, d1_params = run_classification(
    'D1', d1, 'is_canceled',
    drop_cols=['reservation_status', 'reservation_status_date', 'assigned_room_type'],
    sample_n=40000
)
display(d1_balance)
display(d1_metrics)
d1_params

Gia dinh thuat toan D1: Logistic Regression gia dinh quan he gan tuyen tinh sau ma hoa va can scale bien so; Decision Tree khong can scale, bat duoc tuong tac phi tuyen nhung de overfit nen can gioi han `max_depth` va `min_samples_leaf`.

## D3 - US Accidents

Bai toan phan lop: du doan muc do nghiem trong `Severity`. Day la bai toan da lop va mat can bang, vi vay macro precision/recall/F1 quan trong hon accuracy. Cac cot mo ta tu do/ID va cac moc thoi gian ket thuc khong dua truc tiep vao mo hinh de giam nhieu va tranh dung thong tin sau su kien.

In [ ]:
d3_path = RAW / 'D3_us_accidents' / 'US_Accidents_sampled.csv'
if not d3_path.exists():
    files = list((RAW / 'D3_us_accidents').glob('US_Accidents*.csv'))
    if not files:
        raise FileNotFoundError('Thieu D3. Chay bai1-du-lieu-tien-xu-ly/download_data.py hoac dat US_Accidents_sampled.csv vao data/raw/D3_us_accidents')
    d3_path = files[0]
d3 = pd.read_csv(d3_path)
for c in ['Start_Time', 'End_Time', 'Weather_Timestamp']:
    if c in d3.columns:
        d3[c] = pd.to_datetime(d3[c], errors='coerce')
if 'Start_Time' in d3.columns:
    d3['start_hour'] = d3['Start_Time'].dt.hour
    d3['start_dayofweek'] = d3['Start_Time'].dt.dayofweek
    d3['start_month'] = d3['Start_Time'].dt.month
if {'Start_Time', 'End_Time'}.issubset(d3.columns):
    d3['duration_minutes'] = (d3['End_Time'] - d3['Start_Time']).dt.total_seconds().div(60).clip(lower=0, upper=24*60)
keep = [
    'Severity','Start_Lat','Start_Lng','Distance(mi)','Temperature(F)','Humidity(%)','Pressure(in)',
    'Visibility(mi)','Wind_Speed(mph)','Precipitation(in)','Amenity','Bump','Crossing','Give_Way',
    'Junction','No_Exit','Railway','Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal',
    'Sunrise_Sunset','Civil_Twilight','Weather_Condition','Wind_Direction','Side','State','Timezone',
    'start_hour','start_dayofweek','start_month','duration_minutes'
]
keep = [c for c in keep if c in d3.columns]
d3_model = d3[keep].dropna(subset=['Severity'])
print(d3.shape, d3_model.shape)
d3_balance, d3_metrics, d3_params = run_classification(
    'D3', d3_model, 'Severity', drop_cols=[], sample_n=40000
)
display(d3_balance)
display(d3_metrics)
d3_params

Gia dinh thuat toan D3: Logistic Regression dung duong bien tuyen tinh trong khong gian da ma hoa va can class weight vi mat can bang; Decision Tree bat quan he nguong nhu visibility, distance, gio trong ngay va cac co duong, nhung can regularization bang do sau va kich thuoc la toi thieu.

In [ ]:
all_metrics = pd.concat([pd.read_csv(OUT/'D1_metrics.csv'), pd.read_csv(OUT/'D3_metrics.csv')], ignore_index=True)
all_metrics.to_csv(OUT/'classification_metrics_all.csv', index=False)
all_metrics.sort_values(['dataset','f1'], ascending=[True, False])

## Nhan xet ky thuat ngan

- Voi D1, accuracy co the doc duoc vi lop huy/khong huy khong qua cuc doan, nhung F1 van la chi so chinh de so sanh hai mo hinh.
- Voi D3, do mat can bang lop, macro F1 va macro recall quan trong hon accuracy. Neu accuracy cao nhung macro recall thap thi mo hinh dang thien ve lop pho bien.
- Logistic Regression thuong on dinh va de tong quat hoa sau khi scale/one-hot; Decision Tree de giai thich hon va bat duoc nguong phi tuyen, nhung can gioi han do sau de tranh overfit.
- Output da luu trong `bai2-phan-lop/outputs`: bang can bang lop, metrics, sieu tham so tot nhat va confusion matrix cho tung mo hinh.